<a href="https://colab.research.google.com/github/TiaGolyan2508/python-/blob/main/brainTumorSegmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries for medical image processing
# nibabel -> used to read MRI files (.nii/.nii.gz)
# monai -> medical imaging deep learning toolkit
# grad-cam -> for explainable AI visualization
!pip install nibabel
!pip install monai
!pip install torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 17.7 MB/s eta 0:00:00


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Import important libraries

import torch                     # PyTorch deep learning framework
import torch.nn as nn            # neural network layers
import numpy as np               # numerical operations
import nibabel as nib            # to load MRI files
import matplotlib.pyplot as plt  # visualization
from torch.utils.data import Dataset, DataLoader

In [7]:
!ls /content/drive/MyDrive/

'Colab Notebooks'  'Memories '


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls /content/drive/MyDrive/

 BraTS	'Colab Notebooks'  'Memories '


In [3]:
!ls /content/drive/MyDrive/BraTS/

In [4]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"tiya2508","key":"6f413a343154b5994bc4a0da920d3d28"}'}

In [5]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [6]:
!kaggle datasets download -d awsaf49/brats20-dataset-training-validation

Dataset URL: https://www.kaggle.com/datasets/awsaf49/brats20-dataset-training-validation
License(s): CC0-1.0
 99% 4.13G/4.16G [00:53<00:00, 38.9MB/s]
100% 4.16G/4.16G [00:53<00:00, 84.1MB/s]


In [7]:
!unzip brats20-dataset-training-validation.zip -d /content/brats

Archive:  brats20-dataset-training-validation.zip
  inflating: /content/brats/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001/BraTS20_Training_001_flair.nii  
  inflating: /content/brats/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001/BraTS20_Training_001_seg.nii  
  inflating: /content/brats/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001/BraTS20_Training_001_t1.nii  
  inflating: /content/brats/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001/BraTS20_Training_001_t1ce.nii  
  inflating: /content/brats/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001/BraTS20_Training_001_t2.nii  
  inflating: /content/brats/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_002/BraTS20_Training_002_flair.nii  
  inflating: /content/brats/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_002/BraTS20_Training_002_seg.nii  
  inflating: /co

In [8]:
!ls /content/brats

BraTS2020_TrainingData	BraTS2020_ValidationData


In [1]:
# Core deep learning library
import torch
import torch.nn as nn
import torch.nn.functional as F

# Dataset utilities
from torch.utils.data import Dataset, DataLoader

# MRI loading
import nibabel as nib

# Numerical operations
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# File navigation
import os

In [2]:
# This function loads MRI scans stored in .nii.gz format
# BraTS dataset stores all MRI modalities in this format

def load_nifti(path):

    # Load MRI volume
    img = nib.load(path)

    # Convert MRI to numpy array
    data = img.get_fdata()

    return data

In [3]:
class BraTSDataset(Dataset):

    def __init__(self, patient_dirs):

        self.samples = []

        # Loop through each patient folder
        for patient in patient_dirs:

            base = patient

            t1 = load_nifti(os.path.join(base, os.path.basename(base) + "_t1.nii.gz"))
            t2 = load_nifti(os.path.join(base, os.path.basename(base) + "_t2.nii.gz"))
            t1ce = load_nifti(os.path.join(base, os.path.basename(base) + "_t1ce.nii.gz"))
            flair = load_nifti(os.path.join(base, os.path.basename(base) + "_flair.nii.gz"))

            mask = load_nifti(os.path.join(base, os.path.basename(base) + "_seg.nii.gz"))

            # Convert 3D MRI volume into 2D slices
            for i in range(t1.shape[2]):

                img = np.stack([
                    t1[:,:,i],
                    t2[:,:,i],
                    t1ce[:,:,i],
                    flair[:,:,i]
                ], axis=0)

                m = mask[:,:,i]

                self.samples.append((img, m))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        img, mask = self.samples[idx]

        img = torch.tensor(img).float()
        mask = torch.tensor(mask).float().unsqueeze(0)

        return img, mask

In [4]:
class ModalityReliability(nn.Module):

    """
    Novel module that learns the reliability of each MRI modality.
    Instead of treating all modalities equally,
    the model learns which MRI scan is more useful.
    """

    def __init__(self, channels=4):
        super().__init__()

        # 1x1 convolution computes modality importance
        self.conv = nn.Conv2d(channels, channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        weights = self.sigmoid(self.conv(x))

        # Multiply input MRI with reliability weights
        return x * weights

In [5]:
class DoubleConv(nn.Module):

    """
    Standard convolution block used in U-Net.
    Extracts spatial features from MRI images.
    """

    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

In [6]:
class AttentionBlock(nn.Module):

    """
    Attention block focuses the model on tumor regions
    and suppresses irrelevant background information.
    """

    def __init__(self, F_g, F_l, F_int):
        super().__init__()

        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, 1),
            nn.BatchNorm2d(F_int)
        )

        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, 1),
            nn.BatchNorm2d(F_int)
        )

        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, g, x):

        g1 = self.W_g(g)
        x1 = self.W_x(x)

        psi = self.psi(F.relu(g1 + x1))

        return x * psi

In [7]:
class BrainTumorSegmentationModel(nn.Module):

    def __init__(self):
        super().__init__()

        # Novel modality reliability module
        self.modality_attention = ModalityReliability()

        # Encoder
        self.conv1 = DoubleConv(4, 64)
        self.pool = nn.MaxPool2d(2)

        self.conv2 = DoubleConv(64,128)

        # Decoder
        self.up = nn.ConvTranspose2d(128,64,2,stride=2)

        self.att = AttentionBlock(64,64,32)

        self.conv3 = DoubleConv(128,64)

        # Output segmentation layer
        self.out = nn.Conv2d(64,1,1)

    def forward(self, x):

        # Apply modality reliability weighting
        x = self.modality_attention(x)

        # Encoder
        e1 = self.conv1(x)
        e2 = self.conv2(self.pool(e1))

        # Decoder
        d1 = self.up(e2)

        # Apply attention
        e1_att = self.att(d1,e1)

        d1 = torch.cat([d1,e1_att],dim=1)

        d1 = self.conv3(d1)

        # Final tumor segmentation
        return torch.sigmoid(self.out(d1))

In [8]:
def dice_loss(pred, target):

    smooth = 1

    intersection = (pred * target).sum()

    return 1 - ((2 * intersection + smooth) /
               (pred.sum() + target.sum() + smooth))

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BrainTumorSegmentationModel().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

epochs = 10

for epoch in range(epochs):

    for images, masks in train_loader:

        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)

        loss = dice_loss(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch:", epoch, "Loss:", loss.item())

NameError: name 'train_loader' is not defined

In [11]:
# Check if GPU is available in Colab
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# Move model to GPU if available
model = BrainTumorSegmentationModel().to(device)

Using device: cuda


In [13]:
# Path where the BraTS dataset was extracted
data_dir = "/content/brats"

import os

# Collect all patient folders
patients = [os.path.join(data_dir, p) for p in os.listdir(data_dir)]

print("Total patients:", len(patients))

FileNotFoundError: [Errno 2] No such file or directory: '/content/brats'